# Build StratifiedGroupKFold fold-assignment lookup

Produces one lookup table keyed on `(burst, Ind_ID, new_burst)` giving each sample a `fold`.
All sub-windows of a parent burst (`burst_id`) share a fold, so sub-bursts never leak across train/test.
Downstream model scripts join on `(burst, Ind_ID, new_burst)` and build: train = `fold != f`, test = `fold == f`.

## Imports & config

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# ---- SET THESE PATHS ----
LOOKUP_PATH = "../data/raw/annotated_burst_id_lookup_table.csv"
ANNOT_PATH  = "../data/temp/annotations/attributions_merged_majority_outliers.csv"
OUTPUT_PATH = "../data/temp/indices_cv_fold_lookup.csv"

CLASS_COL = "attribution_merged"
REMOVE_VALUE = "Remove"
RANDOM_STATE = 123
K_FOLDS = None 


## Step 1 — Assemble the working table
Join the label onto the lookup spine, drop unmodelled/unlabelled rows so folds cover exactly what the models see.

In [ ]:
lookup = pd.read_csv(LOOKUP_PATH)   # burst, new_burst, Ind_ID, burst_id
annot  = pd.read_csv(ANNOT_PATH)

annot_slim = annot[["Ind_ID", "new_burst", CLASS_COL, "activity"]].drop_duplicates(["Ind_ID", "new_burst"])
work = lookup.merge(annot_slim, on=["Ind_ID", "new_burst"], how="left")

before = len(work)
work = work[work[CLASS_COL].notna() & (work[CLASS_COL] != REMOVE_VALUE)].copy()
print(f"Dropped {before - len(work)} unlabelled/Remove rows; {len(work)} samples remain")
assert work[CLASS_COL].notna().all()

# expected columns present
for c in ["burst", "new_burst", "Ind_ID", "burst_id"]:
    assert c in work.columns, f"missing column: {c}"
work.head()

## Step 2 — Parent-level majority label
Collapse to one row per parent burst `(burst, Ind_ID, burst_id)` and give it the majority class of its samples (ties broken alphabetically for determinism). This is what the stratifier acts on, since whole parents move together.

In [ ]:
def majority_label(s):
    counts = s.value_counts()
    top = counts[counts == counts.max()].index
    return sorted(top)[0]           # deterministic tie-break

parents = (
    work.groupby(["burst", "Ind_ID", "burst_id"])[CLASS_COL]
        .agg(majority_label)
        .reset_index()
        .rename(columns={CLASS_COL: "group_label"})
)
print(f"{len(parents)} parent bursts across {parents['burst'].nunique()} burst datasets")
parents.head()


## Step 3 — Diagnostic: distinct parents per class, choose k
The rarest class sets the ceiling on k. Rule: `K_FOLDS <= min distinct parent bursts for any class in any burst dataset`. Review the table, then set `K_FOLDS` above if you want to override.

In [ ]:
diag = (
    parents.groupby(["burst", "group_label"]).size()
           .rename("n_parents").reset_index()
           .pivot(index="group_label", columns="burst", values="n_parents")
           .fillna(0).astype(int)
)
print("Distinct parent bursts per class (by majority label):")
print(diag)

nonzero = diag.replace(0, np.nan)
floor = int(nonzero.min().min())
# which (class, burst) sets the floor
fl_pos = nonzero.stack().idxmin()
print(f"\nRare-class floor = {floor}  (set by class '{fl_pos[0]}' in {fl_pos[1]})")

recommended_k = min(5, floor)

K_FOLDS_EFFECTIVE = K_FOLDS if K_FOLDS is not None else recommended_k
print(f"Recommended K_FOLDS = {recommended_k}")
assert K_FOLDS_EFFECTIVE <= floor, f"K_FOLDS={K_FOLDS_EFFECTIVE} exceeds rare-class floor {floor}"
print(f"Using K_FOLDS = {K_FOLDS_EFFECTIVE}")


## Step 4 — Assign folds per burst dataset, propagate to samples
Per burst dataset, the rare-class parents (Running, Self-scratching) are first dealt to folds in turn (order shuffled with `RANDOM_STATE`) to guarantee they are spread across folds. StratifiedGroupKFold is then run on the remaining parents (one row per parent, so grouping is trivially satisfied and it behaves as stratified k-fold over parents). Every sample inherits its parent's fold.

In [ ]:
RARE_CLASSES = ["Running", "Self-scratching"]   # round-robined to guarantee spread; everything else stratified

parents["fold"] = -1
rng  = np.random.default_rng(RANDOM_STATE)
sgkf = StratifiedGroupKFold(n_splits=K_FOLDS_EFFECTIVE, shuffle=True, random_state=RANDOM_STATE)

for b, idx in parents.groupby("burst").groups.items():
    sub = parents.loc[idx]
    is_rare = sub["group_label"].isin(RARE_CLASSES)

    # --- rare classes: even round-robin across folds (deterministic, shuffled within class) ---
    for cls in RARE_CLASSES:
        cls_idx = sub.index[sub["group_label"] == cls].to_numpy()
        if len(cls_idx) == 0:
            continue
        cls_idx = rng.permutation(cls_idx)                       # shuffle which parents land where
        parents.loc[cls_idx, "fold"] = np.arange(len(cls_idx)) % K_FOLDS_EFFECTIVE

    # --- everything else: StratifiedGroupKFold on the common-class parents ---
    common = sub.loc[~is_rare]
    groups = (common["Ind_ID"].astype(str) + "_" + common["burst_id"].astype(str)).to_numpy()
    y = common["group_label"].to_numpy()
    X = np.zeros(len(common))
    for fold_id, (_, test_idx) in enumerate(sgkf.split(X, y, groups)):
        parents.loc[common.index[test_idx], "fold"] = fold_id

assert (parents["fold"] >= 0).all(), "some parent bursts were not assigned a fold"

# propagate each parent's fold to all its samples
fold_table = work.merge(
    parents[["burst", "Ind_ID", "burst_id", "group_label", "fold"]],
    on=["burst", "Ind_ID", "burst_id"], how="left",
)
assert fold_table["fold"].notna().all()
fold_table["fold"] = fold_table["fold"].astype(int)
print("Folds assigned (rare classes round-robined, rest stratified).")

## Step 5 — Validate (must pass before saving)

In [ ]:
# 1) leakage: each parent burst confined to one fold
grp_folds = fold_table.groupby(["burst", "Ind_ID", "burst_id"])["fold"].nunique()
assert (grp_folds == 1).all(), "LEAKAGE: a parent burst spans multiple folds"
print("PASS: every parent burst is confined to a single fold")

# 2) coverage: every sample has exactly one valid fold
assert fold_table["fold"].between(0, K_FOLDS_EFFECTIVE - 1).all()
print("PASS: every sample assigned exactly one valid fold")

# 3) per-fold, per-class counts; flag zero cells
for b in sorted(fold_table["burst"].unique()):
    tab = fold_table[fold_table.burst == b].pivot_table(
        index="fold", columns=CLASS_COL, values="new_burst", aggfunc="count", fill_value=0)
    print(f"\n{b}: samples per (fold x class)")
    print(tab)
    zeros = [(int(f), c) for f in tab.index for c in tab.columns if tab.loc[f, c] == 0]
    if zeros:
        print("  NOTE zero (fold, class) cells:", zeros)


## Step 6 — Save

In [ ]:
cols = ["burst", "Ind_ID", "new_burst", "burst_id", CLASS_COL, "activity", "group_label", "fold"]
out = fold_table[cols].rename(columns={CLASS_COL: "behaviour"})
out.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(out)} rows -> {OUTPUT_PATH}")
out.head()


## Downstream contract
In each model script, join data to this table on `(burst, Ind_ID, new_burst)` to attach `fold`, then for fold `f`:
`train = fold != f`, `test = fold == f`. Model scripts write per-fold metrics (used in the stats with fold as a random effect), plus per-fold confusion matrices and per-window out-of-fold predictions. These can be pooled across folds so rare-class metrics stay defined even if a fold lacks a class.

## Class counts and proportions per fold
Run this cell on its own; do not re-run the steps above, because Step 6 would overwrite the lookup.
It reads the saved lookup, i.e. the exact folds every model used, and tabulates the windows of each behaviour per (burst, fold).

In [ ]:
import pandas as pd

fold_lookup = pd.read_csv("../data/temp/indices_cv_fold_lookup.csv")
counts = (
    fold_lookup.groupby(["burst", "fold", "behaviour"]).size()
    .unstack("behaviour", fill_value=0)
)
props = counts.div(counts.sum(axis=1), axis=0).round(3)
counts.to_csv("../data/output/comparison_behaviour/behaviour_class_counts_per_fold.csv")
props.to_csv("../data/output/comparison_behaviour/behaviour_class_props_per_fold.csv")
print(counts)
props